In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:
SEED = 29

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Path to processed data:", PROCESSED_DATA_DIR.resolve())

Konfiguracja gotowa. Ścieżka zapisu: C:\Users\aisia\Desktop\Meisterstück\mgr inz\ver1004\hybrid_LUNAR\data\processed


In [ ]:
def print_unsw_label_summary_pre_split(df_all):
    """Generuje szczegółowe podsumowanie etykiet i kategorii ataków dla UNSW-NB15 przed splitem."""
    df_temp = df_all.copy()
    
    # Standaryzacja attack cat
    if 'attack_cat' in df_temp.columns:
        df_temp['attack_cat'] = df_temp['attack_cat'].astype(str).str.strip().str.title()
        df_temp['attack_cat'] = df_temp['attack_cat'].replace({'Nan': 'Normal', 'Nanol': 'Normal', '': 'Normal'})
        df_temp.loc[df_temp['label'] == 0, 'attack_cat'] = 'Normal'
    else:
        df_temp['attack_cat'] = df_temp['label'].map({0: 'Normal', 1: 'Attack'})

    total_rows = len(df_temp)
    
    # Agregacja per kategoria i etykieta binarna
    summary = (
        df_temp.groupby(['attack_cat', 'label'])
        .size()
        .reset_index(name='Count')
    )
    summary['Share (%)'] = (summary['Count'] / total_rows * 100).round(2)
    summary = summary.sort_values(by='Count', ascending=False).reset_index(drop=True)
    summary.columns = ['Attack Category', 'Label (0/1)', 'Sample Count', 'Share (%)']

    print("\n" + "="*70)
    print(" [UNSW-NB15] PRE-SPLIT DETAILED LABEL BREAKDOWN")
    print("="*70)
    print(summary.to_string(index=False))
    print("-" * 70)
    print(f"TOTAL RAW ROWS LOADED: {total_rows:,}")
    print("="*70 + "\n")


def print_cicids_label_summary_pre_split(df_all, label_col='Label'):
    """Generuje szczegółowe podsumowanie etykiet dla CICIDS 2017 przed splitem."""
    df_temp = df_all.copy()
    
    # Czyszczenie ewentualnych spacji w nazwach etykiet
    df_temp['Label_Clean'] = df_temp[label_col].astype(str).str.strip()
    df_temp['Binary_Label'] = (df_temp['Label_Clean'] != 'BENIGN').astype(int)

    total_rows = len(df_temp)

    summary = (
        df_temp.groupby(['Label_Clean', 'Binary_Label'])
        .size()
        .reset_index(name='Count')
    )
    summary['Share (%)'] = (summary['Count'] / total_rows * 100).round(4)
    summary = summary.sort_values(by='Count', ascending=False).reset_index(drop=True)
    summary.columns = ['Specific Label', 'Binary (0/1)', 'Sample Count', 'Share (%)']

    print("\n" + "="*75)
    print(" [CICIDS 2017] PRE-SPLIT DETAILED LABEL BREAKDOWN")
    print("="*75)
    print(summary.to_string(index=False))
    print("-" * 75)
    print(f"TOTAL RAW ROWS LOADED: {total_rows:,}")
    print("="*75 + "\n")

In [4]:
def split_val_into_tune_and_calib(val_df, label_col="label", calib_fraction=0.5, seed=SEED):
    """
    Dzieli pierwotny zbiór walidacyjny na dwa ROZŁĄCZNE zbiory:
    - val_tune: do optymalizacji hiperparametrów (Optuna)
    - val_calib: do kalibracji progu (threshold selection)
    Zachowuje naturalną proporcję klas (stratified split) bez oversamplingu.
    """
    y = val_df[label_col].values
    stratify = y if len(np.unique(y)) > 1 else None
    
    val_tune_df, val_calib_df = train_test_split(
        val_df,
        test_size=calib_fraction,
        stratify=stratify,
        random_state=seed
    )
    return val_tune_df, val_calib_df

def print_split_summary(dataset_name, train_df, val_tune_df, val_calib_df, test_df, label_col="label"):
    """Wyświetla podsumowanie liczebności i proporcji klas w poszczególnych zbiorach."""
    print(f"\n================ PODSUMOWANIE DANYCH: {dataset_name} ================")
    splits = {
        "Train (Pure Normal)": train_df,
        "Val Tune (Optuna)": val_tune_df,
        "Val Calib (Threshold)": val_calib_df,
        "Test (Frozen Evaluation)": test_df
    }
    
    summary = []
    for name, df in splits.items():
        total = len(df)
        pos = int((df[label_col] == 1).sum()) if label_col in df.columns else 0
        neg = total - pos
        ratio = (pos / total * 100) if total > 0 else 0.0
        summary.append({
            "Split": name,
            "Total Rows": total,
            "Normal (0)": neg,
            "Attack (1)": pos,
            "Attack Ratio (%)": f"{ratio:.2f}%"
        })
    
    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
    print("=" * 68)

In [ ]:
def process_unsw_nb15(raw_dir=RAW_DATA_DIR, output_dir=PROCESSED_DATA_DIR, seed=SEED):
    print("\n[UNSW-NB15] Wczytywanie surowych plików CSV...")
    
    train_raw_path = raw_dir / "UNSW_NB15_training-set.csv"
    test_raw_path = raw_dir / "UNSW_NB15_testing-set.csv"
    
    if not train_raw_path.exists() or not test_raw_path.exists():
        print(f"BŁĄD: Brak plików w {raw_dir}. Upewnij się, że pliki UNSW_NB15_training-set.csv i UNSW_NB15_testing-set.csv istnieją.")
        return

    df_train_raw = pd.read_csv(train_raw_path)
    df_test_raw = pd.read_csv(test_raw_path)

    # Oznaczenie zbiorów przed spójnym kodowaniem One-Hot
    df_train_raw['__split'] = 'train_pool'
    df_test_raw['__split'] = 'test'
    df_all = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

    # =========================================================
    # PODSUMOWANIE PER LABEL PRZED SPLITEM:
    print_unsw_label_summary_pre_split(df_all)

    # Usunięcie zbędnych kolumn identyfikacyjnych i kategorii ataków
    drop_cols = [c for c in ['id', 'attack_cat'] if c in df_all.columns]
    df_all = df_all.drop(columns=drop_cols)

    # Kodowanie zmiennych kategorycznych
    cat_cols = ['proto', 'service', 'state']
    cat_cols = [c for c in cat_cols if c in df_all.columns]
    df_all = pd.get_dummies(df_all, columns=cat_cols)

    # Rozdzielenie z powrotem na pulę treningową i test
    df_train_pool = df_all[df_all['__split'] == 'train_pool'].drop(columns=['__split'])
    df_test = df_all[df_all['__split'] == 'test'].drop(columns=['__split'])

    # Wydzielenie czystego zbioru treningowego (wyłącznie normalny ruch, label == 0)
    train_normal = df_train_pool[df_train_pool['label'] == 0].copy()
    
    # Pula ataków z pliku treningowego
    attack_pool = df_train_pool[df_train_pool['label'] == 1].copy()

    # Przenosimy 20% normalnego ruchu do walidacji
    train_normal_final, val_normal = train_test_split(
        train_normal, test_size=0.20, random_state=seed
    )

    # === ZMIANA: Dopasowanie proporcji ataków w walidacji do zbioru testowego ===
    test_attack_ratio = (df_test['label'] == 1).mean()  # ok. 55.06%
    n_val_normal = len(val_normal)                       # 11 200 próbek
    
    # Wyliczenie ilu ataków potrzebujemy, aby uzyskać taki sam % jak na teście
    n_val_attacks_needed = int(n_val_normal * (test_attack_ratio / (1.0 - test_attack_ratio)))

    # Losowy podbiór ataków (subsampling) z zachowaniem ziarna (seed)
    val_attacks = attack_pool.sample(n=n_val_attacks_needed, random_state=seed)

    # Połączenie ruchu normalnego i zbalansowanych ataków w pełną walidację
    val_full = pd.concat([val_normal, val_attacks], ignore_index=True)

    # Rozdzielenie walidacji na val_tune i val_calib (50/50 stratified)
    val_tune, val_calib = split_val_into_tune_and_calib(val_full, label_col='label', calib_fraction=0.5, seed=seed)

    # Skalowanie (MinMaxScaler fit wyłącznie na czystym treningu)
    feature_cols = [c for c in train_normal_final.columns if c != 'label']
    scaler = MinMaxScaler()
    
    train_normal_final[feature_cols] = scaler.fit_transform(train_normal_final[feature_cols])
    val_tune[feature_cols] = scaler.transform(val_tune[feature_cols])
    val_calib[feature_cols] = scaler.transform(val_calib[feature_cols])
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])

    # Zapis do plików CSV
    train_normal_final.to_csv(output_dir / "UNSW_NB15_train.csv", index=False)
    val_tune.to_csv(output_dir / "UNSW_NB15_val_tune.csv", index=False)
    val_calib.to_csv(output_dir / "UNSW_NB15_val_calib.csv", index=False)
    df_test.to_csv(output_dir / "UNSW_NB15_test.csv", index=False)

    print_split_summary("UNSW-NB15 (Balanced Val)", train_normal_final, val_tune, val_calib, df_test, label_col='label')

process_unsw_nb15()


[UNSW-NB15] Wczytywanie surowych plików CSV...

 [UNSW-NB15] PRE-SPLIT DETAILED LABEL BREAKDOWN
Attack Category  Label (0/1)  Sample Count  Share (%)
         Normal            0         93000      36.09
        Generic            1         58871      22.85
       Exploits            1         44525      17.28
        Fuzzers            1         24246       9.41
            Dos            1         16353       6.35
 Reconnaissance            1         13987       5.43
       Analysis            1          2677       1.04
       Backdoor            1          2329       0.90
      Shellcode            1          1511       0.59
          Worms            1           174       0.07
----------------------------------------------------------------------
TOTAL RAW ROWS LOADED: 257,673


================ PODSUMOWANIE DANYCH: UNSW-NB15 (Balanced Val) ================
                   Split  Total Rows  Normal (0)  Attack (1) Attack Ratio (%)
     Train (Pure Normal)       44800       4480

In [ ]:
def process_cicids2017(raw_dir=RAW_DATA_DIR, output_dir=PROCESSED_DATA_DIR, seed=SEED):
    print("\n[CICIDS 2017] Wczytywanie i przetwarzanie plików...")
    
    # Przykładowe pliki: Monday = ruch czysty, pozostałe dni = ataki + normalny ruch
    monday_path = raw_dir / "Monday-WorkingHours.csv"
    attack_paths = list(raw_dir.glob("*WorkingHours*.csv"))
    attack_paths = [p for p in attack_paths if "Monday" not in p.name]

    if not monday_path.exists() or len(attack_paths) == 0:
        print(f"BŁĄD: Brak plików CICIDS 2017 w {raw_dir}. Wymagany jest plik Monday oraz pliki z ataków.")
        return

    # Wczytanie poniedziałku (Czysty ruch normalny)[cite: 8, 10]
    df_monday = pd.read_csv(monday_path)
    df_monday.columns = df_monday.columns.str.strip()
    
    # Wczytanie ataków z pozostałych dni
    attack_dfs = []
    for p in attack_paths:
        df_tmp = pd.read_csv(p)
        df_tmp.columns = df_tmp.columns.str.strip()
        attack_dfs.append(df_tmp)
    df_attacks_all = pd.concat(attack_dfs, ignore_index=True)

    # Łączenie do spójnego czyszczenia
    df_monday['__split'] = 'monday'
    df_attacks_all['__split'] = 'attacks'
    df_all = pd.concat([df_monday, df_attacks_all], ignore_index=True)


    # Czyszczenie wartości nieskończonych i NaN
    df_all = df_all.replace([np.inf, -np.inf], np.nan).dropna()

    # Usunięcie kolumn nieistotnych/wyciekowych
    drop_cols = [c for c in ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp'] if c in df_all.columns]
    df_all = df_all.drop(columns=drop_cols)

    # Etykietowanie: BENIGN -> 0, Atak -> 1
    label_col = 'Label' if 'Label' in df_all.columns else 'label'
    df_all[label_col] = (df_all[label_col].astype(str).str.upper() != 'BENIGN').astype(int)


    # =========================================================
    # STATYSTYKI PRZED SPLITEM -FYI
    print_cicids_label_summary_pre_split(df_all)

    # Kodowanie zmiennych kategorycznych jeśli istnieją
    cat_cols = [c for c in df_all.columns if df_all[c].dtype == object and c not in ['__split', label_col]]
    if cat_cols:
        df_all = pd.get_dummies(df_all, columns=cat_cols)

    # Rozdzielenie
    df_monday_clean = df_all[df_all['__split'] == 'monday'].drop(columns=['__split'])
    df_attacks_clean = df_all[df_all['__split'] == 'attacks'].drop(columns=['__split'])

    # 1. Zbiór Treningowy: Wyłącznie ruch czysty z poniedziałku[cite: 8, 10]
    train_normal = df_monday_clean[df_monday_clean[label_col] == 0].copy()

    # 2. Podział danych z ataków na Walidację i Test (np. 50/50 stratified)
    y_attacks = df_attacks_clean[label_col].values
    val_full, df_test = train_test_split(
        df_attacks_clean, test_size=0.50, stratify=y_attacks, random_state=seed
    )

    # 3. Rozdzielenie Walidacji na Val Tune i Val Calib (50/50 stratified)[cite: 1, 10]
    val_tune, val_calib = split_val_into_tune_and_calib(val_full, label_col=label_col, calib_fraction=0.5, seed=seed)

    # Skalowanie (MinMaxScaler fit wyłącznie na czystym treningu)[cite: 8]
    feature_cols = [c for c in train_normal.columns if c != label_col]
    scaler = MinMaxScaler()
    
    train_normal[feature_cols] = scaler.fit_transform(train_normal[feature_cols])
    val_tune[feature_cols] = scaler.transform(val_tune[feature_cols])
    val_calib[feature_cols] = scaler.transform(val_calib[feature_cols])
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])

    # Zapis do CSV
    train_normal.to_csv(output_dir / "CICIDS2017_train.csv", index=False)
    val_tune.to_csv(output_dir / "CICIDS2017_val_tune.csv", index=False)
    val_calib.to_csv(output_dir / "CICIDS2017_val_calib.csv", index=False)
    df_test.to_csv(output_dir / "CICIDS2017_test.csv", index=False)

    print_split_summary("CICIDS 2017", train_normal, val_tune, val_calib, df_test, label_col=label_col)

process_cicids2017()


[CICIDS 2017] Wczytywanie i przetwarzanie plików...

 [CICIDS 2017] PRE-SPLIT DETAILED LABEL BREAKDOWN
Specific Label  Binary (0/1)  Sample Count  Share (%)
             0             1       2271320    80.3189
             1             1        556556    19.6811
---------------------------------------------------------------------------
TOTAL RAW ROWS LOADED: 2,827,876


================ PODSUMOWANIE DANYCH: CICIDS 2017 ================
                   Split  Total Rows  Normal (0)  Attack (1) Attack Ratio (%)
     Train (Pure Normal)      529481      529481           0            0.00%
       Val Tune (Optuna)      574598      435459      139139           24.22%
   Val Calib (Threshold)      574599      435460      139139           24.21%
Test (Frozen Evaluation)     1149198      870920      278278           24.21%
